# Notebook 44 — Residual Flow Fixed Points and Phase Diagram

This notebook upgrades the residual-flow analysis from Notebook 43 into an explicit **dynamical-structure** study.

We now ask whether the learned residual flow behaves like a structured system of the form

\[
\frac{dr}{dc} = g(r,c)
\]

where:

- `r` = residual coordinate
- `c` = condition coordinate
- `g(r,c)` = learned local flow field

The goal is to determine whether the residual field exhibits:

- **nullclines / fixed regions** (`g(r,c) \approx 0`)
- **local stability or instability** via `\partial g / \partial r`
- **basins of attraction**
- **forward vs backward asymmetry**
- **path dependence / non-conservative structure**
- **scale dependence across k = 3, 5, 7**

## Continuity with earlier notebooks

- **40A**: no residual under symmetric architecture
- **40B**: residual can be forced
- **41**: residual is not a shared aligned subspace
- **42**: residual behaves like transport / deformation
- **43**: residual behaves like a local nonlinear flow

Therefore:

> Notebook 44 extracts the **geometric and dynamical structure** of that flow.

In [ ]:

# Core imports
import os
import re
import json
import math
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.metrics import r2_score
from sklearn.neighbors import KernelDensity

warnings.filterwarnings("ignore")

# Plot style: keep simple and portable
plt.rcParams["figure.figsize"] = (9, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 11

RNG = np.random.default_rng(44)

## Data assumptions

This notebook is written to be robust to small schema differences. It will try to load a tabular dataset containing flow samples or pointwise diagnostics from Notebook 43 or an equivalent export.

### Preferred columns

A minimally useful table should contain:

- `system` — e.g. `entropy`, `unevenness`
- `task` — e.g. `zeta_vs_gue`, `zeta_vs_poisson`
- `forcing_mode` — e.g. `capacity_gap`, `feature_gap`, `condition_gap`
- `k` — e.g. `3`, `5`, `7`
- `flow_mode` — e.g. `nonlinear`, `linear`
- `condition_coord` — local condition coordinate `c`
- `residual` — local residual coordinate `r`
- `predicted_flow` — estimated local flow `g(r,c)`

### Helpful optional columns

- `next_residual`
- `delta_condition`
- `sample_id`
- `path_id`
- `window_id`
- `jacobian_norm`
- `integration_error`

If your exported table uses different column names, update the alias map in the next cell.

If no external export is found, the notebook can fall back to a **synthetic residual-flow dataset** that preserves the intended structure (systems, tasks, forcing modes, k values, and nonlinear flow behavior) so the analysis remains runnable end-to-end.

In [ ]:

# ---- Column alias handling ---------------------------------------------------
COLUMN_ALIASES = {
    "system": ["system"],
    "task": ["task"],
    "forcing_mode": ["forcing_mode", "forcing", "gap_mode"],
    "k": ["k", "window_k", "window_size"],
    "flow_mode": ["flow_mode", "model_type", "flow_type"],
    "condition_coord": ["condition_coord", "condition", "c", "cond_coord"],
    "residual": ["residual", "r", "residual_value"],
    "predicted_flow": ["predicted_flow", "flow", "g_hat", "pred_flow", "predicted_delta_residual_per_condition"],
    "next_residual": ["next_residual", "r_next"],
    "delta_condition": ["delta_condition", "dc", "delta_c"],
    "sample_id": ["sample_id", "id"],
    "path_id": ["path_id", "trajectory_id", "curve_id"],
    "window_id": ["window_id", "window"],
    "jacobian_norm": ["jacobian_norm", "jac_norm"],
    "integration_error": ["integration_error", "int_error"]
}

def standardize_columns(df, alias_map=COLUMN_ALIASES):
    rename = {}
    lower_map = {c.lower(): c for c in df.columns}
    for canonical, aliases in alias_map.items():
        for alias in aliases:
            if alias in df.columns:
                rename[alias] = canonical
                break
            if alias.lower() in lower_map:
                rename[lower_map[alias.lower()]] = canonical
                break
    out = df.rename(columns=rename).copy()
    return out

def ensure_required_columns(df, required=("system","task","forcing_mode","k","flow_mode","condition_coord","residual","predicted_flow")):
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns after alias resolution: {missing}")

In [ ]:

# ---- Data loading ------------------------------------------------------------
SEARCH_PATHS = [
    ".",
    "./data",
    "./exports",
    "./results",
    "./artifacts",
    "./notebooks",
    "../data",
    "../exports",
    "../results",
    "../artifacts",
]

PREFERRED_PATTERNS = [
    "*43*.csv", "*43*.parquet", "*43*.feather", "*43*.pkl",
    "*residual*flow*.csv", "*residual*flow*.parquet",
    "*flow_field*.csv", "*flow_field*.parquet",
    "*residual*.csv", "*residual*.parquet"
]

def find_candidate_files(search_paths=SEARCH_PATHS, patterns=PREFERRED_PATTERNS):
    hits = []
    for base in search_paths:
        for pat in patterns:
            hits.extend(glob.glob(str(Path(base) / pat)))
    # unique, sorted by shortest path then alpha
    hits = sorted(set(hits), key=lambda x: (len(x), x))
    return hits

def load_table(path):
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix == ".feather":
        return pd.read_feather(path)
    if suffix == ".pkl":
        return pd.read_pickle(path)
    raise ValueError(f"Unsupported file type: {path}")

candidate_files = find_candidate_files()
print("Candidate files:")
for p in candidate_files[:30]:
    print(" -", p)

In [ ]:
# Set this manually if the auto-detected file is not the right export.
DATA_PATH = candidate_files[0] if candidate_files else None
USE_SYNTHETIC_FALLBACK = DATA_PATH is None
print("Selected DATA_PATH:", DATA_PATH)
print("USE_SYNTHETIC_FALLBACK:", USE_SYNTHETIC_FALLBACK)

def make_synthetic_residual_flow_dataset(rng=None):
    """
    Build a synthetic dataset with the same required schema as Notebook 43 exports.

    The construction intentionally creates:
    - a dominant stable basin,
    - curved nonlinear flow,
    - modest forward/backward asymmetry,
    - task / system / forcing / k dependence.
    """
    rng = np.random.default_rng(44 if rng is None else rng)

    systems = ["entropy", "unevenness"]
    tasks = ["zeta_vs_gue", "zeta_vs_poisson"]
    forcing_modes = ["capacity_gap", "feature_gap", "condition_gap"]
    ks = [3, 5, 7]
    flow_modes = ["linear", "nonlinear"]

    rows = []
    sample_id = 0

    def forcing_strength(name):
        return {
            "capacity_gap": 0.85,
            "feature_gap": 0.95,
            "condition_gap": 1.15,
        }.get(name, 1.0)

    def task_shift(name):
        return {
            "zeta_vs_gue": 0.10,
            "zeta_vs_poisson": -0.05,
        }.get(name, 0.0)

    def system_shift(name):
        return {
            "entropy": 0.12,
            "unevenness": -0.08,
        }.get(name, 0.0)

    def k_shift(k):
        return {3: -0.18, 5: 0.00, 7: 0.10}.get(int(k), 0.0)

    for system in systems:
        for task in tasks:
            for forcing_mode in forcing_modes:
                for k in ks:
                    for flow_mode in flow_modes:
                        n_paths = 14
                        n_steps = 42

                        for path_idx in range(n_paths):
                            c_vals = np.linspace(-1.25, 1.25, n_steps)
                            c_vals += rng.normal(0.0, 0.015, size=n_steps)
                            c_vals = np.sort(c_vals)

                            # spread initial residuals so basin plots are visible
                            r = rng.uniform(-1.35, 1.35)

                            for step_idx, c in enumerate(c_vals):
                                fs = forcing_strength(forcing_mode)
                                ts = task_shift(task)
                                ss = system_shift(system)
                                ksft = k_shift(k)

                                # stable center moves with c and metadata
                                anchor = (
                                    0.55 * np.tanh(1.35 * c)
                                    + 0.15 * np.sin(2.1 * c + ts)
                                    + 0.12 * ss
                                    + 0.08 * ksft
                                )

                                # restoring component
                                lam = (0.85 + 0.22 * fs) * (1.00 + 0.08 * (k - 5) / 2.0)

                                if flow_mode == "linear":
                                    nonlinear_term = 0.0
                                else:
                                    nonlinear_term = (
                                        0.24 * fs * np.tanh(1.25 * r - 0.7 * c)
                                        - 0.12 * (r - anchor) ** 3
                                        + 0.08 * np.sin(1.8 * c + 0.65 * r)
                                    )

                                # asymmetry / weak path dependence proxy
                                directional_bias = 0.05 * np.sign(c + 1e-9) * (0.4 + fs)
                                curvature = 0.07 * np.sin((k / 2.0) * c) * (r - anchor)

                                g = (
                                    -lam * (r - anchor)
                                    + nonlinear_term
                                    + directional_bias
                                    + curvature
                                    + rng.normal(0.0, 0.045 + 0.01 * (flow_mode == "nonlinear"))
                                )

                                if step_idx < n_steps - 1:
                                    dc = float(c_vals[step_idx + 1] - c)
                                else:
                                    dc = float(np.median(np.diff(c_vals)))

                                r_next = r + g * dc + rng.normal(0.0, 0.01)

                                rows.append({
                                    "system": system,
                                    "task": task,
                                    "forcing_mode": forcing_mode,
                                    "k": int(k),
                                    "flow_mode": flow_mode,
                                    "condition_coord": float(c),
                                    "residual": float(r),
                                    "predicted_flow": float(g),
                                    "next_residual": float(r_next),
                                    "delta_condition": float(dc),
                                    "sample_id": int(sample_id),
                                    "path_id": int(path_idx),
                                    "window_id": int(step_idx),
                                    "jacobian_norm": float(abs(-lam + 0.15 * np.cos(1.8 * c + 0.65 * r))),
                                    "integration_error": float(abs(rng.normal(0.02 + 0.01 * (flow_mode == "nonlinear"), 0.01))),
                                })

                                sample_id += 1
                                r = r_next

    out = pd.DataFrame(rows)
    return out

if USE_SYNTHETIC_FALLBACK:
    print("No candidate export found. Generating synthetic fallback dataset...")
    df_raw = make_synthetic_residual_flow_dataset(RNG)
    DATA_SOURCE_NOTE = "synthetic_fallback"
else:
    df_raw = load_table(DATA_PATH)
    DATA_SOURCE_NOTE = str(DATA_PATH)

df = standardize_columns(df_raw)

ensure_required_columns(df)

# basic type cleanup
for col in ["k", "condition_coord", "residual", "predicted_flow"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["system","task","forcing_mode","k","flow_mode","condition_coord","residual","predicted_flow"]).copy()

print("Data source:", DATA_SOURCE_NOTE)
print("Shape:", df.shape)
display(df.head())

## Experiment configuration

We retain continuity with the previous notebooks:

- systems: `entropy`, `unevenness`
- tasks: `zeta_vs_gue`, `zeta_vs_poisson`
- forcing modes: `capacity_gap`, `feature_gap`, `condition_gap`
- window sizes: `k = 3, 5, 7`

For the most detailed diagnostics, we prioritize:

- `forcing_mode = condition_gap`
- `flow_mode = nonlinear`
- `task = zeta_vs_gue`

but compare outward to the rest.

In [ ]:

# ---- Basic summaries ---------------------------------------------------------
summary = (
    df.groupby(["system","task","forcing_mode","k","flow_mode"])
      .size()
      .rename("n")
      .reset_index()
      .sort_values(["system","task","forcing_mode","k","flow_mode"])
)
display(summary)

focus_defaults = {
    "system": summary["system"].mode().iat[0] if len(summary) else "entropy",
    "task": "zeta_vs_gue" if "zeta_vs_gue" in set(df["task"]) else sorted(df["task"].unique())[0],
    "forcing_mode": "condition_gap" if "condition_gap" in set(df["forcing_mode"]) else sorted(df["forcing_mode"].unique())[0],
    "k": 5 if 5 in set(df["k"].astype(int)) else int(sorted(df["k"].unique())[0]),
    "flow_mode": "nonlinear" if "nonlinear" in set(df["flow_mode"]) else sorted(df["flow_mode"].unique())[0],
}
focus_defaults

In [ ]:

# Choose the focal slice here if needed
FOCUS = focus_defaults.copy()

focus_df = df[
    (df["system"] == FOCUS["system"]) &
    (df["task"] == FOCUS["task"]) &
    (df["forcing_mode"] == FOCUS["forcing_mode"]) &
    (df["k"].astype(int) == int(FOCUS["k"])) &
    (df["flow_mode"] == FOCUS["flow_mode"])
].copy()

print("FOCUS:", FOCUS)
print("Focus shape:", focus_df.shape)
display(focus_df.head())

## Utility functions

We build a small analysis layer for:

1. phase-diagram aggregation
2. nullcline extraction
3. local stability estimation
4. trajectory integration
5. path-dependence scoring
6. energy-like potential probing

In [ ]:

# ---- Analysis utilities ------------------------------------------------------
def quantile_edges(x, bins=25, eps=1e-9):
    x = np.asarray(x)
    qs = np.linspace(0, 1, bins + 1)
    edges = np.quantile(x, qs)
    edges[0] -= eps
    edges[-1] += eps
    # ensure strictly increasing
    for i in range(1, len(edges)):
        if edges[i] <= edges[i-1]:
            edges[i] = edges[i-1] + eps
    return edges

def build_phase_grid(sub, c_bins=28, r_bins=28, min_count=8):
    d = sub[["condition_coord","residual","predicted_flow"]].dropna().copy()
    c_edges = quantile_edges(d["condition_coord"], c_bins)
    r_edges = quantile_edges(d["residual"], r_bins)

    d["c_bin"] = pd.cut(d["condition_coord"], bins=c_edges, labels=False, include_lowest=True)
    d["r_bin"] = pd.cut(d["residual"], bins=r_edges, labels=False, include_lowest=True)

    g = (
        d.groupby(["c_bin","r_bin"])
         .agg(
             mean_flow=("predicted_flow","mean"),
             std_flow=("predicted_flow","std"),
             count=("predicted_flow","size"),
             mean_c=("condition_coord","mean"),
             mean_r=("residual","mean"),
         )
         .reset_index()
    )

    g["std_flow"] = g["std_flow"].fillna(0.0)
    g["is_reliable"] = g["count"] >= min_count

    return g, c_edges, r_edges

def pivot_grid(g, value):
    p = g.pivot(index="r_bin", columns="c_bin", values=value).sort_index(ascending=True)
    return p

def detect_nullcline_bins(g, abs_flow_quantile=0.15, max_std_quantile=0.7):
    reliable = g[g["is_reliable"]].copy()
    if len(reliable) == 0:
        out = g.copy()
        out["near_null"] = False
        return out
    flow_thr = reliable["mean_flow"].abs().quantile(abs_flow_quantile)
    std_thr = reliable["std_flow"].quantile(max_std_quantile)
    out = g.copy()
    out["near_null"] = (
        out["is_reliable"] &
        (out["mean_flow"].abs() <= flow_thr) &
        (out["std_flow"] <= std_thr)
    )
    return out

def detect_zero_crossings_by_cslice(g):
    # look for sign changes along r within each c_bin
    flags = []
    for c_bin, sub in g.sort_values(["c_bin","r_bin"]).groupby("c_bin"):
        sub = sub[sub["is_reliable"]].sort_values("r_bin")
        prev_sign = None
        prev_idx = None
        for idx, row in sub.iterrows():
            sign = np.sign(row["mean_flow"])
            if sign == 0:
                flags.append(idx)
            elif prev_sign is not None and sign != prev_sign:
                flags.extend([prev_idx, idx])
            prev_sign = sign
            prev_idx = idx
    out = g.copy()
    out["zero_crossing_bin"] = out.index.isin(flags)
    return out

def estimate_local_stability(sub, g, neighborhood_frac_c=0.08, neighborhood_frac_r=0.08, min_points=18):
    # local slope d g / d r in neighborhoods around grid cell centers
    c_lo, c_hi = sub["condition_coord"].quantile([0.05, 0.95])
    r_lo, r_hi = sub["residual"].quantile([0.05, 0.95])
    dc = max(1e-12, neighborhood_frac_c * (c_hi - c_lo))
    dr = max(1e-12, neighborhood_frac_r * (r_hi - r_lo))

    rows = []
    for _, row in g.iterrows():
        cc = row["mean_c"]
        rr = row["mean_r"]
        hood = sub[
            (sub["condition_coord"].between(cc - dc, cc + dc)) &
            (sub["residual"].between(rr - dr, rr + dr))
        ][["residual","predicted_flow"]].dropna()

        if len(hood) < min_points or hood["residual"].nunique() < 5:
            slope = np.nan
            intercept = np.nan
            r2 = np.nan
        else:
            X = hood[["residual"]].values
            y = hood["predicted_flow"].values
            lr = LinearRegression().fit(X, y)
            slope = float(lr.coef_[0])
            intercept = float(lr.intercept_)
            r2 = float(lr.score(X, y))

        rows.append({
            "c_bin": row["c_bin"],
            "r_bin": row["r_bin"],
            "dg_dr": slope,
            "stability_r2": r2,
            "stable": (slope < 0) if pd.notnull(slope) else False,
            "unstable": (slope > 0) if pd.notnull(slope) else False,
        })

    stab = pd.DataFrame(rows)
    return g.merge(stab, on=["c_bin","r_bin"], how="left")

def build_flow_interpolator(sub, neighborhood_frac_c=0.08, neighborhood_frac_r=0.08, min_points=20, min_unique=8):
    # local weighted average model for g(r,c), intentionally simple and robust
    d = sub[["condition_coord","residual","predicted_flow"]].dropna().copy()
    c_rng = max(1e-12, d["condition_coord"].quantile(0.95) - d["condition_coord"].quantile(0.05))
    r_rng = max(1e-12, d["residual"].quantile(0.95) - d["residual"].quantile(0.05))
    dc = neighborhood_frac_c * c_rng
    dr = neighborhood_frac_r * r_rng

    def ghat(r, c):
        hood = d[
            (d["condition_coord"].between(c - dc, c + dc)) &
            (d["residual"].between(r - dr, r + dr))
        ]
        if len(hood) < min_points or hood["residual"].nunique() < min_unique:
            # broader fallback
            hood = d[
                (d["condition_coord"].between(c - 2*dc, c + 2*dc)) &
                (d["residual"].between(r - 2*dr, r + 2*dr))
            ]
        if len(hood) == 0:
            return np.nan
        dist = ((hood["condition_coord"] - c) / max(dc, 1e-12))**2 + ((hood["residual"] - r) / max(dr, 1e-12))**2
        w = np.exp(-0.5 * dist.values)
        y = hood["predicted_flow"].values
        return float(np.average(y, weights=w))

    return ghat

def integrate_flow(ghat, c_grid, r0, direction="forward", step_scale=1.0, clip_quantiles=None):
    c_grid = np.asarray(c_grid)
    if direction == "backward":
        c_seq = c_grid[::-1]
    else:
        c_seq = c_grid.copy()

    rs = [float(r0)]
    cs = [float(c_seq[0])]
    valid = True
    for i in range(len(c_seq)-1):
        c_now = c_seq[i]
        c_next = c_seq[i+1]
        dc = (c_next - c_now) * step_scale
        r_now = rs[-1]
        g = ghat(r_now, c_now)
        if not np.isfinite(g):
            valid = False
            break
        r_next = r_now + g * dc
        if clip_quantiles is not None:
            lo, hi = clip_quantiles
            r_next = float(np.clip(r_next, lo, hi))
        rs.append(float(r_next))
        cs.append(float(c_next))

    out = pd.DataFrame({"condition_coord": cs, "residual": rs})
    out["direction"] = direction
    out["valid"] = valid
    return out

def path_dependence_score(ghat, c_grid, r0_values, clip_quantiles):
    # compare direct vs split integration around midpoint
    mid = len(c_grid) // 2
    left = c_grid[:mid+1]
    right = c_grid[mid:]

    rows = []
    for r0 in r0_values:
        direct = integrate_flow(ghat, c_grid, r0, "forward", clip_quantiles=clip_quantiles)
        if not direct["valid"].iat[0]:
            continue

        split1 = integrate_flow(ghat, left, r0, "forward", clip_quantiles=clip_quantiles)
        if len(split1) == 0:
            continue
        r_mid = split1["residual"].iloc[-1]
        split2 = integrate_flow(ghat, right, r_mid, "forward", clip_quantiles=clip_quantiles)

        if len(split2) == 0 or not split2["valid"].iat[0]:
            continue

        r_direct = direct["residual"].iloc[-1]
        r_split = split2["residual"].iloc[-1]
        rows.append({"r0": r0, "terminal_direct": r_direct, "terminal_split": r_split, "abs_gap": abs(r_direct - r_split)})

    out = pd.DataFrame(rows)
    score = float(out["abs_gap"].mean()) if len(out) else np.nan
    return out, score

def build_energy_like_potential(g, by_c=True):
    # For each c_bin, integrate -g dr over residual bins to form an energy-like profile.
    curves = []
    for c_bin, sub in g.sort_values(["c_bin","r_bin"]).groupby("c_bin"):
        sub = sub.sort_values("mean_r")
        if len(sub) < 3:
            continue
        r = sub["mean_r"].values
        flow = sub["mean_flow"].values
        # cumulative trapezoidal integral of -g dr
        E = [0.0]
        for i in range(1, len(r)):
            dr = r[i] - r[i-1]
            avg = -0.5 * (flow[i] + flow[i-1])
            E.append(E[-1] + avg * dr)
        tmp = sub[["c_bin","r_bin","mean_c","mean_r"]].copy()
        tmp["energy_like"] = E
        curves.append(tmp)
    if not curves:
        return pd.DataFrame(columns=["c_bin","r_bin","mean_c","mean_r","energy_like"])
    return pd.concat(curves, ignore_index=True)

## Experiment A — Fixed points and nullclines

We aggregate the flow field on a `(c, r)` grid, estimate mean flow per cell, and mark candidate fixed regions in two ways:

1. **near-null bins**: low `|g|` with controlled variance  
2. **zero-crossing bins**: sign changes in `g` along `r` slices for fixed `c`

This is stronger than thresholding alone and gets closer to a true nullcline view.

In [ ]:

g0, c_edges, r_edges = build_phase_grid(focus_df, c_bins=28, r_bins=28, min_count=8)
g1 = detect_nullcline_bins(g0, abs_flow_quantile=0.15, max_std_quantile=0.70)
g1 = detect_zero_crossings_by_cslice(g1)

print("Grid cells:", len(g1))
print("Reliable cells:", int(g1["is_reliable"].sum()))
print("Near-null cells:", int(g1["near_null"].sum()))
print("Zero-crossing cells:", int(g1["zero_crossing_bin"].sum()))

display(g1.head())

In [ ]:

# Plot: mean flow phase diagram with nullcline overlays
p_flow = pivot_grid(g1, "mean_flow")
plt.figure(figsize=(10, 7))
plt.imshow(p_flow.values, origin="lower", aspect="auto")
plt.colorbar(label="Mean predicted flow")
plt.title("Mean flow phase diagram")
plt.xlabel("condition bin")
plt.ylabel("residual bin")

# overlays
for _, row in g1[g1["near_null"]].iterrows():
    plt.scatter(row["c_bin"], row["r_bin"], s=22, marker="o", facecolors="none", edgecolors="white")
for _, row in g1[g1["zero_crossing_bin"]].iterrows():
    plt.scatter(row["c_bin"], row["r_bin"], s=30, marker="x")

plt.show()

## Experiment B — Local stability

We estimate local stability via a local slope fit:

\[
\frac{\partial g}{\partial r}
\]

Interpretation:

- **negative slope** → restoring, stable
- **positive slope** → repelling, unstable

This is the notebook’s most important quantitative test.

In [ ]:

g2 = estimate_local_stability(focus_df, g1, neighborhood_frac_c=0.08, neighborhood_frac_r=0.08, min_points=18)

stable_count = int(g2["stable"].fillna(False).sum())
unstable_count = int(g2["unstable"].fillna(False).sum())
print("Stable cells:", stable_count)
print("Unstable cells:", unstable_count)

display(g2[["c_bin","r_bin","mean_c","mean_r","mean_flow","dg_dr","stability_r2","stable","unstable"]].head())

In [ ]:

# Plot: stability map
p_stab = pivot_grid(g2, "dg_dr")
plt.figure(figsize=(10, 7))
plt.imshow(p_stab.values, origin="lower", aspect="auto")
plt.colorbar(label="Local slope d g / d r")
plt.title("Local stability map")
plt.xlabel("condition bin")
plt.ylabel("residual bin")
plt.show()

plt.figure(figsize=(8, 5))
valid_slopes = g2["dg_dr"].dropna()
plt.hist(valid_slopes.values, bins=40)
plt.axvline(0.0, linestyle="--")
plt.title("Distribution of local slopes")
plt.xlabel("d g / d r")
plt.ylabel("Count")
plt.show()

## Experiment C — Full phase diagram

We combine:

- mean flow
- nullcline candidates
- stability sign

to build an interpretable dynamical portrait rather than just a predictive heatmap.

In [ ]:

# Combined phase portrait scatter on physical coordinates
plt.figure(figsize=(10, 7))
reliable = g2[g2["is_reliable"]].copy()
sc = plt.scatter(reliable["mean_c"], reliable["mean_r"], c=reliable["mean_flow"], s=65, alpha=0.85)
plt.colorbar(sc, label="Mean predicted flow")

stable = reliable[reliable["stable"].fillna(False)]
unstable = reliable[reliable["unstable"].fillna(False)]
nullish = reliable[reliable["near_null"] | reliable["zero_crossing_bin"]]

if len(stable):
    plt.scatter(stable["mean_c"], stable["mean_r"], marker="s", s=35, facecolors="none", edgecolors="black", label="stable")
if len(unstable):
    plt.scatter(unstable["mean_c"], unstable["mean_r"], marker="^", s=35, facecolors="none", edgecolors="red", label="unstable")
if len(nullish):
    plt.scatter(nullish["mean_c"], nullish["mean_r"], marker="x", s=40, label="nullcline candidate")

plt.title("Residual flow phase portrait")
plt.xlabel("condition coordinate c")
plt.ylabel("residual coordinate r")
plt.legend(loc="best")
plt.show()

## Experiment D — Basin structure

We now integrate the learned field across condition coordinate and ask:

- where does the residual end up from different initial conditions?
- is there one basin or several?
- do thresholds separate convergence classes?

This converts the local field into a basin-of-attraction picture.

In [ ]:

ghat = build_flow_interpolator(focus_df, neighborhood_frac_c=0.08, neighborhood_frac_r=0.08)

c_grid = np.linspace(focus_df["condition_coord"].quantile(0.05), focus_df["condition_coord"].quantile(0.95), 120)
r_lo, r_hi = focus_df["residual"].quantile([0.05, 0.95])
r0_values = np.linspace(r_lo, r_hi, 45)
clip_bounds = (focus_df["residual"].quantile(0.01), focus_df["residual"].quantile(0.99))

traj_rows = []
terminals = []
for r0 in r0_values:
    tr = integrate_flow(ghat, c_grid, r0, direction="forward", clip_quantiles=clip_bounds)
    tr["r0"] = r0
    traj_rows.append(tr)
    if len(tr) and tr["valid"].iat[0]:
        terminals.append({"r0": r0, "terminal_residual": tr["residual"].iloc[-1]})

traj_df = pd.concat(traj_rows, ignore_index=True)
terminal_df = pd.DataFrame(terminals)

display(terminal_df.head())

In [ ]:

# basin plot: trajectories
plt.figure(figsize=(10, 7))
for r0, sub in traj_df.groupby("r0"):
    plt.plot(sub["condition_coord"], sub["residual"], alpha=0.35)
plt.title("Integrated trajectories across initial residuals")
plt.xlabel("condition coordinate c")
plt.ylabel("residual")
plt.show()

# terminal state map
plt.figure(figsize=(9, 5))
plt.plot(terminal_df["r0"], terminal_df["terminal_residual"], marker="o")
plt.title("Terminal residual as a function of initial residual")
plt.xlabel("initial residual r0")
plt.ylabel("terminal residual")
plt.show()

In [ ]:

# cluster terminal states for convergence classes (if structure exists)
if len(terminal_df) >= 6:
    n_clusters = 2 if len(terminal_df) >= 10 else 1
    km = KMeans(n_clusters=n_clusters, n_init=10, random_state=44)
    terminal_df["basin_class"] = km.fit_predict(terminal_df[["terminal_residual"]])
else:
    terminal_df["basin_class"] = 0

plt.figure(figsize=(9, 5))
plt.scatter(terminal_df["r0"], terminal_df["terminal_residual"], c=terminal_df["basin_class"], s=50)
plt.title("Basin classes from terminal residuals")
plt.xlabel("initial residual r0")
plt.ylabel("terminal residual")
plt.show()

display(terminal_df.head())

## Experiment E — Comparison across k

We test whether the geometry changes across `k = 3, 5, 7`.

A meaningful scale effect can show up as changes in:

- number of fixed / null cells
- stable fraction
- unstable fraction
- typical slope magnitude
- path dependence score

In [ ]:

def summarize_slice(sub):
    g0, _, _ = build_phase_grid(sub, c_bins=24, r_bins=24, min_count=6)
    g1 = detect_nullcline_bins(g0, abs_flow_quantile=0.15, max_std_quantile=0.70)
    g1 = detect_zero_crossings_by_cslice(g1)
    g2 = estimate_local_stability(sub, g1, neighborhood_frac_c=0.08, neighborhood_frac_r=0.08, min_points=14)

    ghat = build_flow_interpolator(sub, neighborhood_frac_c=0.08, neighborhood_frac_r=0.08)
    c_grid = np.linspace(sub["condition_coord"].quantile(0.05), sub["condition_coord"].quantile(0.95), 100)
    r_lo, r_hi = sub["residual"].quantile([0.10, 0.90])
    r0_vals = np.linspace(r_lo, r_hi, 25)
    clip_bounds = (sub["residual"].quantile(0.01), sub["residual"].quantile(0.99))
    _, pd_score = path_dependence_score(ghat, c_grid, r0_vals, clip_bounds)

    return {
        "n_cells": len(g2),
        "reliable_cells": int(g2["is_reliable"].sum()),
        "null_cells": int((g2["near_null"] | g2["zero_crossing_bin"]).sum()),
        "stable_cells": int(g2["stable"].fillna(False).sum()),
        "unstable_cells": int(g2["unstable"].fillna(False).sum()),
        "mean_abs_slope": float(g2["dg_dr"].abs().dropna().mean()) if g2["dg_dr"].notna().any() else np.nan,
        "path_dependence_score": pd_score,
    }

rows = []
for (system, task, forcing_mode, k, flow_mode), sub in df.groupby(["system","task","forcing_mode","k","flow_mode"]):
    if len(sub) < 80:
        continue
    out = summarize_slice(sub)
    out.update({
        "system": system,
        "task": task,
        "forcing_mode": forcing_mode,
        "k": int(k),
        "flow_mode": flow_mode,
    })
    rows.append(out)

k_summary = pd.DataFrame(rows).sort_values(["system","task","forcing_mode","flow_mode","k"])
display(k_summary.head(20))

In [ ]:

# Focused grouped view for selected system/task/forcing/flow across k
k_focus = k_summary[
    (k_summary["system"] == FOCUS["system"]) &
    (k_summary["task"] == FOCUS["task"]) &
    (k_summary["forcing_mode"] == FOCUS["forcing_mode"]) &
    (k_summary["flow_mode"] == FOCUS["flow_mode"])
].sort_values("k")

display(k_focus)

plt.figure(figsize=(9,5))
plt.plot(k_focus["k"], k_focus["null_cells"], marker="o", label="null cells")
plt.plot(k_focus["k"], k_focus["stable_cells"], marker="s", label="stable cells")
plt.plot(k_focus["k"], k_focus["unstable_cells"], marker="^", label="unstable cells")
plt.title("Fixed/stable/unstable structure across k")
plt.xlabel("k")
plt.ylabel("count")
plt.legend()
plt.show()

plt.figure(figsize=(9,5))
plt.plot(k_focus["k"], k_focus["path_dependence_score"], marker="o")
plt.title("Path dependence score across k")
plt.xlabel("k")
plt.ylabel("score")
plt.show()

## Experiment F — Forward vs backward integration

This is a critical asymmetry test.

If forward integration is consistently better than backward integration, the field behaves more like a **directed / dissipative deformation** than a conservative reversible portrait.

In [ ]:

def compare_forward_backward(sub):
    ghat = build_flow_interpolator(sub, neighborhood_frac_c=0.08, neighborhood_frac_r=0.08)
    c_grid = np.linspace(sub["condition_coord"].quantile(0.05), sub["condition_coord"].quantile(0.95), 120)
    r_lo, r_hi = sub["residual"].quantile([0.10, 0.90])
    r0_vals = np.linspace(r_lo, r_hi, 30)
    clip_bounds = (sub["residual"].quantile(0.01), sub["residual"].quantile(0.99))

    rows = []
    for r0 in r0_vals:
        fwd = integrate_flow(ghat, c_grid, r0, "forward", clip_quantiles=clip_bounds)
        bwd = integrate_flow(ghat, c_grid, r0, "backward", clip_quantiles=clip_bounds)
        if len(fwd) and len(bwd):
            rows.append({
                "r0": r0,
                "forward_terminal": fwd["residual"].iloc[-1],
                "backward_terminal": bwd["residual"].iloc[-1],
                "terminal_gap": abs(fwd["residual"].iloc[-1] - bwd["residual"].iloc[-1]),
                "forward_valid": bool(fwd["valid"].iat[0]),
                "backward_valid": bool(bwd["valid"].iat[0]),
            })
    return pd.DataFrame(rows)

fb_df = compare_forward_backward(focus_df)
display(fb_df.head())

In [ ]:

plt.figure(figsize=(9,5))
plt.plot(fb_df["r0"], fb_df["forward_terminal"], marker="o", label="forward terminal")
plt.plot(fb_df["r0"], fb_df["backward_terminal"], marker="s", label="backward terminal")
plt.title("Forward vs backward terminal states")
plt.xlabel("initial residual r0")
plt.ylabel("terminal residual")
plt.legend()
plt.show()

plt.figure(figsize=(9,5))
plt.plot(fb_df["r0"], fb_df["terminal_gap"], marker="o")
plt.title("Forward/backward asymmetry")
plt.xlabel("initial residual r0")
plt.ylabel("|forward - backward|")
plt.show()

## Experiment G — Energy-like potential probe

To test whether the field is approximately conservative, we build an **energy-like profile** along residual slices:

\[
E(r \mid c) \approx \int -g(r,c) \, dr
\]

This is not a full proof of a global potential, but it gives a useful diagnostic:

- smooth single-well structure → more restoring
- jagged or inconsistent structure across `c` → more non-conservative / path-dependent

In [ ]:

energy_df = build_energy_like_potential(g2)
display(energy_df.head())

plt.figure(figsize=(10, 7))
for c_bin, sub in energy_df.groupby("c_bin"):
    if c_bin % 3 != 0:  # thin for readability
        continue
    plt.plot(sub["mean_r"], sub["energy_like"], alpha=0.6)
plt.title("Energy-like profiles across condition slices")
plt.xlabel("residual")
plt.ylabel("energy-like potential")
plt.show()

## Summary table

For each configuration, we report:

- null / fixed-region count
- stable and unstable counts
- mean slope magnitude
- path-dependence score

You can extend this with additional quality columns from Notebook 43 if those exports are available.

In [ ]:

def verdict_row(row):
    null_cells = row.get("null_cells", np.nan)
    stable_cells = row.get("stable_cells", np.nan)
    unstable_cells = row.get("unstable_cells", np.nan)
    pd_score = row.get("path_dependence_score", np.nan)

    if pd.notnull(null_cells) and pd.notnull(stable_cells):
        if stable_cells > unstable_cells and stable_cells > 0:
            if pd.notnull(pd_score) and pd_score < np.nanmedian(k_summary["path_dependence_score"]):
                return "single stable basin"
            return "stable basin with path dependence"
        if unstable_cells > stable_cells and unstable_cells > 0:
            return "weak / unstable phase structure"
        if null_cells >= 4 and stable_cells >= 2:
            return "multi-region nonlinear flow"
    return "inconclusive"

if len(k_summary):
    k_summary["verdict"] = k_summary.apply(verdict_row, axis=1)

display(k_summary.sort_values(["system","task","forcing_mode","flow_mode","k"]).reset_index(drop=True))

## Interpretation guide

Use the following logic:

### Single stable basin
- one dominant fixed region
- mostly negative local slope nearby
- moderate forward/backward agreement
- low path-dependence score

### Multi-basin nonlinear flow
- multiple nullcline regions
- terminal-state clustering
- stability changes across regions

### Path-dependent transport
- clear direct vs split mismatch
- strong forward/backward asymmetry
- inconsistent energy-like profiles across condition slices

### Weak phase structure
- unstable or noisy fixed regions
- little stable area
- poor integration coherence

## Expected readout

Based on the prior notebook chain, a likely outcome is:

- `condition_gap + nonlinear` gives the clearest structure
- one dominant restoring basin appears
- the field is curved rather than linear
- some path dependence remains
- the system is therefore **not fully conservative**

That would support the interpretation:

> residual flow is a nonlinear, weakly dissipative dynamical system with a dominant attractor and scale-dependent structure.

## Next notebook

A natural Notebook 45 would be:

**45_governing_equation_learning_for_residual_flow.ipynb**

where the goal is to move from descriptive phase structure to an explicit learned surrogate for `g(r,c)`, including symbolic or low-complexity approximations if the field is regular enough.